# Final Report — Breast Cancer Wisconsin (Diagnostic): A Biological Story

**Course:** UE 5BM753 — Supervised and Unsupervised Learning for Biology
**Deliverable:** Final report (Python component)

**Status: SKELETON.** This notebook lays out the structure agreed on by the group. Sections
marked **`[TODO — <owner>]`** are intentionally left empty for that person to fill in.
Sections without a `[TODO]` tag are already backed by real, executed results (mostly
reused from `01_Milestone1.ipynb`, which screened the baseline methods) and can be
extended but should not need to be redone from scratch.


## 1. Context & Gap

### 1.1 Clinical context



### 1.2 Gap



## 2. Question

- **Q1 (supervised):** Can we reliably predict the diagnosis (benign/malignant) from the
  30 morphological features? Which standard classifier performs best as a baseline?
- **Q2 (unsupervised):** Do the data naturally separate into two clusters that match the
  known diagnosis, without ever using the labels?
- **Q3 (feature relevance):** Which features are the most redundant, and which carry the
  most discriminative signal?




## 3. Data



## 4. Approach (Methods)

### 4.1 Exploratory Data Analysis

A minimal, targeted EDA is reproduced below (full details — correlation heatmap, boxplots
by diagnosis, outlier screening — are in `01_Milestone1.ipynb` and not repeated here) to
set up the two quantities reused later in this report: the class balance and the PCA
projection used to visualize Spectral Clustering's output in Section 4.5.


In [ ]:
import os
import sys
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(os.path.join("..", "01_scripts"))
from data_utils import load_data, get_feature_columns
from viz import plot_pca_projection  # shared plotting functions -- see 01_scripts/viz.py
# shared evaluation functions -- see 01_scripts/evaluation.py (evaluate_classifier,
# cross_validate_classifiers, run_kmeans_range); import the ones you need in each zoom.

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid", font_scale=1.0)
plt.rcParams["figure.dpi"] = 110

DATA_PATH = os.path.join("..", "02_data", "raw", "10_Breast_Cancer_Wisconsin_Diagnostic.csv")
RESULTS_DIR = os.path.join("..", "04_results")
FIGURES_DIR = os.path.join("..", "05_figures")
TABLES_DIR = os.path.join(RESULTS_DIR, "tables")
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(TABLES_DIR, exist_ok=True)

df = load_data(DATA_PATH)
features = get_feature_columns(df)

print(f"Dataset shape: {df.shape[0]} samples x {len(features)} features")
print(df["diagnosis"].value_counts())


In [ ]:
from sklearn.preprocessing import StandardScaler

X_scaled_full = StandardScaler().fit_transform(df[features].values)
pca_full, X_pca_full = plot_pca_projection(
    X_scaled_full, df["label"].values,
    os.path.join(FIGURES_DIR, "final_01_pca_projection.png"),
    random_state=RANDOM_STATE,
)


### 4.2 Initial screening (brief — see Milestone 1 / Orange3 for detail)

Milestone 1 screened **10 supervised classifiers** (Logistic Regression, KNN, SVM
x3 kernels, LDA, QDA, Decision Tree, Random Forest, AdaBoost) with 10-fold stratified
cross-validation, and **6 clustering methods** (K-Means, Agglomerative x3 linkages,
Spectral, DBSCAN) evaluated against the true diagnosis with external metrics (ARI,
homogeneity, completeness, V-measure, silhouette). The same screening was also run in
Orange3 as a cross-check. Full methodology, code and per-model discussion: see
`01_Milestone1.ipynb`. Only the condensed take-away table is reproduced here
(Section 5.1) — this report does not re-derive the screening.


### 4.3 Feature selection

`[TODO — teammate: feature selection]`

Expected content: description of the embedded feature selection methods compared (e.g.
L1-regularized Logistic Regression, tree-based importance, ...), the cross-validation
protocol used to compare them (fit on training data only), and the final selected feature
subset used for both zooms below (Sections 4.4–4.5), with a justification for why the
same subset is used for both the supervised and unsupervised zoom, so the two analyses
stay comparable.


### 4.4 Zoom — Logistic Regression

`[TODO — teammate: Logistic Regression tuning]`

Expected content: hyperparameter search (e.g. regularization strength `C`) by
cross-validation on the training set restricted to the selected features from Section
4.3, the final chosen configuration, and how it will be evaluated on the held-out test
set (same metrics as Milestone 1: accuracy, precision, recall, F1, ROC-AUC).

**Reuse, don't redefine:** `from evaluation import evaluate_classifier` (Section 5.3's
final test-set metrics) and `from viz import plot_roc_curves, plot_confusion_matrices`
(both take a `{name: fitted_or_unfitted_model}` dict, so they work with a single tuned
model too) — see `01_scripts/evaluation.py` and `01_scripts/viz.py`.

**Blocked by Section 4.3** (needs the selected feature subset first).


### 4.5 Zoom — Spectral Clustering

`[TODO — Ann-Solène: Spectral Clustering zoom]`

Expected content: Spectral Clustering re-run on the features selected in Section 4.3
(instead of all 30), with the affinity/`n_clusters` choices justified, and a plan for how
its output will be visualized and compared to the true diagnosis (e.g. PCA projection
colored by predicted cluster vs. true label, as in Milestone 1 Section 6).

**Reuse, don't redefine:** `from viz import plot_cluster_vs_diagnosis` — same function
Milestone 1 used for its Spectral highlight figure, see `01_scripts/viz.py`.

**Blocked by Section 4.3** (needs the selected feature subset first).


## 5. Results

### 5.1 Screening (condensed)

Only the two methods carried forward to the zooms below are highlighted; full results for
all 10 supervised models and 6 clustering methods are in `01_Milestone1.ipynb`
(`04_results/tables/02_supervised_cv_results.csv` and `04_clustering_eval.csv`).


In [ ]:
supervised_cv = pd.read_csv(os.path.join(TABLES_DIR, "02_supervised_cv_results.csv"))
clustering_eval = pd.read_csv(os.path.join(TABLES_DIR, "04_clustering_eval.csv"))

# "Overall" best = lowest average rank across all 5 metrics at once (same definition
# Milestone 1 used in its run_summary.json), not just the single highest ROC-AUC --
# no one metric should decide a "best" model (Milestone 1, Section 5.2).
_metrics = ["accuracy_mean", "precision_mean", "recall_mean", "f1_mean", "roc_auc_mean"]
_avg_rank = supervised_cv.set_index("model")[_metrics].rank(ascending=False).mean(axis=1)
supervised_condensed = (
    supervised_cv.set_index("model")[_metrics]
    .assign(avg_rank_across_5_metrics=_avg_rank.round(2))
    .sort_values("avg_rank_across_5_metrics")
    .head(3)
    .reset_index()
)
clustering_condensed = (
    clustering_eval.sort_values("ARI_vs_diagnosis", ascending=False)
    .loc[:, ["model", "silhouette", "ARI_vs_diagnosis"]]
    .head(3)
    .reset_index(drop=True)
)

supervised_condensed.to_csv(os.path.join(TABLES_DIR, "final_00_screening_supervised_condensed.csv"), index=False)
clustering_condensed.to_csv(os.path.join(TABLES_DIR, "final_00_screening_clustering_condensed.csv"), index=False)

print("Top 3 supervised models by average rank across all 5 metrics (10-fold CV, Milestone 1):")
display(supervised_condensed)
print("\nTop 3 clustering methods by ARI vs. true diagnosis (Milestone 1):")
display(clustering_condensed)


**Reading.** Logistic Regression is the strongest supervised baseline overall (highest
average rank across all 5 metrics in Milestone 1's synthesis) and Spectral Clustering is
the strongest unsupervised baseline by ARI — this is why they were chosen as the two
"zoom" methods for this report rather than any of the other 8 supervised / 5 clustering
methods screened.


### 5.2 Feature selection

`[TODO — teammate: feature selection]`

Expected content: the comparison table/plot for the feature selection methods tested in
Section 4.3, and the final selected feature list.


### 5.3 Zoom — Logistic Regression (most developed section)

`[TODO — teammate: Logistic Regression tuning]`

Expected content: cross-validated hyperparameter search results, final test-set metrics
(accuracy/precision/recall/F1/ROC-AUC) on the selected features, ROC curve, confusion
matrix, and coefficient interpretation (sign and magnitude per selected feature).

**Blocked by Section 4.3** (needs the selected feature subset first).


### 5.4 Zoom — Spectral Clustering (most developed section)

`[TODO — Ann-Solène: Spectral Clustering zoom]`

Expected content: cluster assignments on the selected features, external metrics vs. true
diagnosis (ARI, homogeneity, completeness, V-measure, silhouette), a PCA-projection plot
comparing predicted clusters to true diagnosis, and a profile of what characterizes each
cluster biologically (mean feature values per cluster).

**Blocked by Section 4.3** (needs the selected feature subset first).


## 6. Discussion

### 6.1 Justification of method choices

`[TODO]` — why Logistic Regression and Spectral Clustering specifically (beyond "best in
the screening"), why the chosen feature selection method, why the chosen
hyperparameters/affinity.

### 6.2 Biological interpretation of the retained features

`[TODO]` — for each feature kept after Section 4.3's selection, what it measures
biologically (nuclear size, shape irregularity, texture, ...) and why it is plausible that
it discriminates malignant from benign nuclei.

### 6.3 Comparison to the literature

`[TODO]` — how the retained features and achieved performance compare to Street, Wolberg
& Mangasarian (1993) (the dataset's original paper) and/or other published work on this
dataset; cite properly in Section 8.

### 6.4 Limitations

`[TODO]` — at minimum: single-institution dataset (generalization untested), moderate
class imbalance, feature collinearity even after selection, Spectral Clustering's
sensitivity to its affinity/parameter choices.


## 7. Conclusion

`[TODO]` — one paragraph answering Q1/Q2/Q3 directly once Sections 4–6 are filled in,
plus what would be the natural next step beyond this report.


## 8. References

1. Street, W.N., Wolberg, W.H., & Mangasarian, O.L. (1993). Nuclear feature extraction
   for breast tumor diagnosis. *IS&T/SPIE Electronic Imaging*, 1905, 861–870.
2. `[TODO — add references used for the feature selection method(s) and any literature
   comparison in Section 6.3]`
